# Lakeside E+ multi-resolution calibration (viewer)

**Heavy EnergyPlus / calib via CLI** — this notebook only reads campaign ledgers and validation JSON.

```powershell
$env:LAKESIDE_SITE_ROOT="C:\Users\ben\OneDrive\Desktop\testing\sp_creekside"
python -u scripts\validate_eplus_multires.py --plots
python -u scripts\eplus_calibrate_multires.py --stage all
```

Physics: IdealLoads + fixed-COP (filename `gshp` is naming only). No optimizer.

In [ ]:
from pathlib import Path
import json, os
import pandas as pd

SITE = Path(os.environ.get("LAKESIDE_SITE_ROOT", r"C:\Users\ben\OneDrive\Desktop\testing\sp_creekside"))
APP = Path("..").resolve() if (Path("..") / "ml").exists() else Path(".").resolve()
camps_dir = SITE / "eplus" / "campaigns"
camps = sorted(camps_dir.glob("multires_*")) if camps_dir.exists() else []
summary_path = camps[-1] / "summary.json" if camps else APP / "ml" / "artifacts" / "eplus_campaigns" / "latest_summary.json"
print("summary:", summary_path)
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.is_file() else {}
summary

In [ ]:
overall = summary.get("validation_overall") or {}
ranking = summary.get("ranking") or {}
pd.DataFrame([
    {"metric": "monthly_pass", "value": overall.get("monthly_pass")},
    {"metric": "hourly_pass", "value": overall.get("hourly_pass")},
    {"metric": "recommendation_allowed", "value": overall.get("recommendation_allowed")},
    {"metric": "blocker", "value": overall.get("blocker_reason")},
    {"metric": "hourly_distance", "value": ranking.get("hourly_distance")},
])

In [ ]:
# Diagnostics: schedule / peak / alignment questions
summary.get("diagnostics") or {}